#### 1. Create the table: **sme_p_business_plan_lbspaymentschedule** in Pre-Landed Layer

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import trim, col

In [0]:
def load_landed_internal_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

In [0]:
run_date = '2026-02-05'
date_filter = f"businessdate = {run_date!r}"
catalog_landed = '`d-cdh`.landed'
cluster_columns = ["BusinessDate"]
landed_internal_table_path = f"{catalog_landed}.sme_p_business_plan_lbspaymentschedule"

In [0]:
# ============================== business_plan_payments_schedule ============================

business_plan_payments_schedule_extra_filters = None

if business_plan_payments_schedule_extra_filters is not None:
    business_plan_payments_schedule_extra_filters.append(date_filter)
    business_plan_payments_schedule_final_filters = " AND ".join(
        business_plan_payments_schedule_extra_filters
        )
else:
    business_plan_payments_schedule_final_filters = date_filter

business_plan_payments_schedule_selected_fields = [
            "ID",
            "LBSParentID",
            "InstallmentSN",
            "InstallmentDate",
            "InstallmentTotalAmount",
            "InstallmentCapitalAmount",
            "InstallmentChargeAmount",
            "InstallmentN128Amount",
            "InstallmentTotalChargeAmount",
            "InstallmentRemainingCapitalAmount",
            "InstallmentNPVAmount",
            "InstallmentNPVCalculationDate",
            "CreationDate",
            "CreationUser",
            "IsActive",
            "BusinessDate"
        ]

business_plan_payments_schedule = load_landed_internal_delta_table(
            spark,
            catalog_landed,  
            "cepalplatform_SmeBpsmLbsPaymentSchedule",
            business_plan_payments_schedule_final_filters,
            business_plan_payments_schedule_selected_fields,
        )

# ============================== filter(s) ============================

business_plan_payments_schedule = business_plan_payments_schedule.withColumn("IsActive", business_plan_payments_schedule.IsActive.cast("Int"))
sme_business_plan_payments_schedule = business_plan_payments_schedule.filter(business_plan_payments_schedule.IsActive == 1)

# ========================= final landed - internal table ============================

sme_business_plan_payments_schedule_df = sme_business_plan_payments_schedule.select(
    sme_business_plan_payments_schedule.ID,
    sme_business_plan_payments_schedule.LBSParentID,
    sme_business_plan_payments_schedule.InstallmentSN,
    sme_business_plan_payments_schedule.InstallmentDate,
    sme_business_plan_payments_schedule.InstallmentTotalAmount,
    sme_business_plan_payments_schedule.InstallmentCapitalAmount,
    sme_business_plan_payments_schedule.InstallmentChargeAmount,
    sme_business_plan_payments_schedule.InstallmentN128Amount,
    sme_business_plan_payments_schedule.InstallmentTotalChargeAmount,
    sme_business_plan_payments_schedule.InstallmentRemainingCapitalAmount,
    sme_business_plan_payments_schedule.InstallmentNPVAmount,
    sme_business_plan_payments_schedule.InstallmentNPVCalculationDate,
    sme_business_plan_payments_schedule.CreationDate,
    sme_business_plan_payments_schedule.CreationUser,
    sme_business_plan_payments_schedule.IsActive,
    sme_business_plan_payments_schedule.BusinessDate
)
